In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
CACHE_DIR = BASE_DIR / "LSTM_cache_TL/Transformer_exp"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:
Build monthly datasets for sequence datasets. 

In [ ]:
SOURCE_REGIONS = [  # regions for "foundation model"
    'CH',
    'NOR',
    'ISL',
    'FR',
]
TARGET_REGIONS = ['SJM']

paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
}

run_flag_by_code = {
    'CH': False,
    'NOR': False,
    'ISL': False,
    'FR': False,
    'SJM': False,
}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

res_xreg_by_source = {
    region: monthly_cache[region]
    for region in SOURCE_REGIONS + TARGET_REGIONS
}

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

### Training glaciers:

#### Automatic picking:

In [ ]:
# remap cache keys to what the function expects
res_xreg_remapped = {
    region: {
        "df_train": res_xreg_by_source[region]['data_monthly'],
        "df_test": res_xreg_by_source[region]['data_monthly_aug'],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_all = build_global_scalers_multi_source_simple(
    res_xreg_remapped,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_xreg_remapped,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(f"blur_m={blur_m:.4f}  blur_s={blur_s:.4f}  blur_joint={blur_joint:.4f}")

#### Split glaciers of training sets:

In [ ]:
RUN_GLACIER_RANK = False

for mode, topo_extra_cols, save_name in [
    ("joint", [], "glacier_ranking_sjm_joint.csv"),
    ("topo", ["ELEVATION_DIFFERENCE"], "glacier_ranking_sjm_topo.csv"),
]:
    save_path = BASE_DIR / save_name

    if RUN_GLACIER_RANK:
        df_ranked = rank_glaciers_by_distance_to_target(
            res_xreg_by_source=res_xreg_by_source,
            training_regions=SOURCE_REGIONS,
            test_region='SJM',
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            blur_joint=blur_joint,
            seed=cfg.seed,
            mode=mode,
            topo_extra_cols=topo_extra_cols,
        )
        df_ranked.to_csv(save_path, index=False)
        print(f"Saved {mode} ranking -> {save_path}")
    else:
        df_ranked = pd.read_csv(save_path)
        print(f"Loaded {mode} ranking from {save_path}")

    if mode == "joint":
        df_ranked_sjm_joint = df_ranked
    else:
        df_ranked_sjm_topo = df_ranked

# --- comparison ---
joint_top10 = set(df_ranked_sjm_joint.head(10)['glacier'])
topo_top10 = set(df_ranked_sjm_topo.head(10)['glacier'])
print(
    f"\nTop-10 overlap (joint ∩ topo): {len(joint_top10 & topo_top10)} glaciers"
)
print(f"Joint only : {sorted(joint_top10 - topo_top10)}")
print(f"Topo only  : {sorted(topo_top10 - joint_top10)}")

## Train transformers:

In [ ]:
gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))

# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[0]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

lstm_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

In [ ]:
FRACS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.4]
N_RANDOM_SEEDS = 10

# --- 1. Build glacier subsets for both rankings ---
gl_subsets_sjm = {}

for mode, df_ranked in [
    ("joint", df_ranked_sjm_joint),
    ("topo", df_ranked_sjm_topo),
]:
    gl_subsets_sjm[mode] = build_glacier_subsets(
        df_ranked=df_ranked,
        fracs=FRACS,
        n_random_seeds=N_RANDOM_SEEDS,
        seed=cfg.seed,
    )
    print(f"\nSubsets built for SJM {mode}")

# convenience aliases
gl_subsets_sjm_joint = gl_subsets_sjm["joint"]
gl_subsets_sjm_topo = gl_subsets_sjm["topo"]

# --- 2. Dummy splits ---
dummy_splits = {
    region: {
        "holdout_glaciers": [],
        "pool_glaciers": []
    }
    for region in SOURCE_REGIONS
}

# --- 3. SJM test dataset (full, zero-shot) ---
ds_sjm_test = build_combined_LSTM_dataset(
    df_loss=res_xreg_by_source['SJM']['data_monthly'],
    df_full=res_xreg_by_source['SJM']['data_monthly_aug'],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['SJM']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['SJM']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)
print(f"SJM test: {len(ds_sjm_test)} sequences")

In [ ]:
# --- 3. Train transformer for each subset ---
TRAIN_FLAG = True
MODEL_DATE = "2026-05-21"
CACHE_DIR_RANK = CACHE_DIR / "ranking_sjm"
os.makedirs(CACHE_DIR_RANK, exist_ok=True)

ranking_results_sjm = []

for ranking_label, df_ranked, gl_subsets_cur in [
    ("ranked_by_SJM_joint", df_ranked_sjm_joint, gl_subsets_sjm_joint),
    ("ranked_by_SJM_topo", df_ranked_sjm_topo, gl_subsets_sjm_topo),
]:
    print(f"\n{'='*60}")
    print(f"  Ranking: {ranking_label}")

    mode = ranking_label.split("_")[-1]
    models_dir = BASE_DIR / "models/ranking_sjm" / mode
    os.makedirs(models_dir, exist_ok=True)

    for pct, subset in gl_subsets_cur.items():
        print(f"\n  Fraction: {pct}%")

        strategies = {
            "closest": subset['closest'],
            **{
                f"random_{s['seed_idx']}": s['glaciers']
                for s in subset['random']
            },
        }

        for strategy_name, glaciers in strategies.items():
            print(f"\n  --- {strategy_name} ({len(glaciers)} glaciers) ---")

            assets = build_assets_from_glacier_list(
                glaciers=glaciers,
                df_ranked=df_ranked,
                res_xreg_by_source=res_xreg_by_source,
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                cfg=cfg,
                cache_path=str(
                    CACHE_DIR_RANK /
                    f"assets_{mode}_{pct}pct_{strategy_name}.joblib"),
                force_recompute=False,
                months_head_pad=months_head_pad,
                months_tail_pad=months_tail_pad,
                src_regions=SOURCE_REGIONS,
            )

            # --- train both model types ---
            for model_type, params, model_label in [
                ("transformer", best_params_gs, "transformer"),
                ("lstm", lstm_params, "lstm"),
            ]:
                model, model_path, info = train_or_load_one_source_model(
                    cfg=cfg,
                    key=f"{pct}pct_{strategy_name}",
                    lstm_assets=assets,
                    best_params=params,
                    device=device,
                    models_dir=models_dir,
                    prefix=f"{model_type}_rank",
                    train_flag=TRAIN_FLAG,
                    force_retrain=False,
                    epochs=150,
                    date=MODEL_DATE,
                    model_type=model_type,
                    verbose=False,
                )

                ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                    assets["ds_train"])
                ds_scaler.make_loaders(
                    train_idx=assets["train_idx"],
                    val_idx=assets["val_idx"],
                    fit_and_transform=True,
                    seed=cfg.seed,
                    verbose=False,
                )
                ds_sjm_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                    ds_sjm_test)
                sjm_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                    ds_sjm_copy,
                    ds_scaler,
                    batch_size=128,
                    seed=cfg.seed,
                )
                metrics, _ = model.evaluate_with_preds(device, sjm_dl,
                                                       ds_sjm_copy)

                is_random = strategy_name.startswith("random")
                print(
                    f"    [{model_label}] RMSE_a={metrics['RMSE_annual']:.3f}  "
                    f"RMSE_w={metrics['RMSE_winter']:.3f}  "
                    f"R2_a={metrics['R2_annual']:.3f}")

                ranking_results_sjm.append({
                    "ranking_target":
                    ranking_label,
                    "model":
                    model_label,
                    "pct":
                    pct,
                    "strategy":
                    "random" if is_random else strategy_name,
                    "seed_idx":
                    int(strategy_name.split("_")[1]) if is_random else None,
                    "n_glaciers":
                    len(glaciers),
                    "n_sequences":
                    len(assets["ds_train"]),
                    **{
                        k: round(v, 3)
                        for k, v in metrics.items()
                    },
                })

    # --- full baselines ---
    print(f"\n  Fraction: 100%")
    full_assets = build_assets_from_glacier_list(
        glaciers=df_ranked_sjm_joint['glacier'].tolist(),
        df_ranked=df_ranked_sjm_joint,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(CACHE_DIR_RANK / "assets_100pct_full.joblib"),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=SOURCE_REGIONS,
    )

    for model_type, params, label in [
        ("lstm", lstm_params, "lstm_full"),
        ("transformer", best_params_gs, "transformer_full"),
    ]:
        model, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key="100pct_full",
            lstm_assets=full_assets,
            best_params=params,
            device=device,
            models_dir=models_dir,
            prefix=f"{model_type}_rank",
            train_flag=TRAIN_FLAG,
            force_retrain=False,
            epochs=150,
            date=MODEL_DATE,
            model_type=model_type,
            verbose=False,
        )

        ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            full_assets["ds_train"])
        ds_scaler.make_loaders(
            train_idx=full_assets["train_idx"],
            val_idx=full_assets["val_idx"],
            fit_and_transform=True,
            seed=cfg.seed,
            verbose=False,
        )
        ds_sjm_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ds_sjm_test)
        sjm_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_sjm_copy,
            ds_scaler,
            batch_size=128,
            seed=cfg.seed,
        )
        metrics, _ = model.evaluate_with_preds(device, sjm_dl, ds_sjm_copy)

        print(f"  {label}: RMSE_a={metrics['RMSE_annual']:.3f}  "
              f"RMSE_w={metrics['RMSE_winter']:.3f}")

        ranking_results_sjm.append({
            "ranking_target": ranking_label,
            "model": label.replace("_full", ""),
            "pct": 100,
            "strategy": label,
            "seed_idx": None,
            "n_glaciers": len(df_ranked_sjm_joint),
            "n_sequences": len(full_assets["ds_train"]),
            **{
                k: round(v, 3)
                for k, v in metrics.items()
            },
        })

# --- save ---
df_ranking_results_sjm = pd.DataFrame(ranking_results_sjm)
df_ranking_results_sjm.to_csv(BASE_DIR / "ranking_sjm_results.csv",
                              index=False)
print(
    df_ranking_results_sjm.groupby(
        ["ranking_target", "model", "pct",
         "strategy"]).mean(numeric_only=True).round(3))

In [ ]:
# SJM — 2 plots (joint + topo), each showing transformer vs lstm
for ranking_label in ["ranked_by_SJM_joint", "ranked_by_SJM_topo"]:
    plot_ranking_results_extended_(
        df_results=df_ranking_results_sjm,
        ranking_label=ranking_label,
        test_region="SJM",
        source_regions=SOURCE_REGIONS,
        n_rand_seeds=N_RANDOM_SEEDS,
        model_label="transformer",  # primary; lstm overlapped automatically
        save_path=BASE_DIR /
        f"ranking_sjm_{ranking_label}_transformer_vs_lstm",
    )

In [ ]:
# --- four plots ---
for model_label in ["transformer", "lstm"]:
    for ranking_label in ["ranked_by_SJM_joint", "ranked_by_SJM_topo"]:
        fig = plot_ranking_results_extended(
            df_results=df_ranking_results_sjm,
            ranking_label=ranking_label,
            test_region="SJM",
            source_regions=SOURCE_REGIONS,
            n_rand_seeds=N_RANDOM_SEEDS,
            model_label=model_label,
            save_path=BASE_DIR / f"ranking_sjm_{ranking_label}_{model_label}",
        )

In [ ]:
FRACS_TO_PLOT = [10, 15, 20, 30]

for mode, df_ranked, gl_subsets_cur in [
    ("joint", df_ranked_sjm_joint, gl_subsets_sjm_joint),
    ("topo", df_ranked_sjm_topo, gl_subsets_sjm_topo),
]:
    models_dir = BASE_DIR / "models/ranking_sjm" / mode
    ranking_plot_configs = []

    for pct in FRACS_TO_PLOT:

        # --- closest ---
        assets_close = build_assets_from_glacier_list(
            glaciers=gl_subsets_cur[pct]['closest'],
            df_ranked=df_ranked,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(CACHE_DIR_RANK /
                           f"assets_{mode}_{pct}pct_closest.joblib"),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            src_regions=SOURCE_REGIONS,
        )
        model_close, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f"{pct}pct_closest",
            lstm_assets=assets_close,
            best_params=best_params_gs,
            device=device,
            models_dir=models_dir,
            prefix="transformer_rank",
            train_flag=False,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        ranking_plot_configs.append(
            (f"Closest {pct}% ({mode})", model_close, assets_close))

        # --- random seed 0 ---
        assets_rnd = build_assets_from_glacier_list(
            glaciers=gl_subsets_cur[pct]['random'][0]['glaciers'],
            df_ranked=df_ranked,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(CACHE_DIR_RANK /
                           f"assets_{mode}_{pct}pct_random_0.joblib"),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            src_regions=SOURCE_REGIONS,
        )
        model_rnd, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f"{pct}pct_random_0",
            lstm_assets=assets_rnd,
            best_params=best_params_gs,
            device=device,
            models_dir=models_dir,
            prefix="transformer_rank",
            train_flag=False,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        ranking_plot_configs.append(
            (f"Random {pct}% seed 0 ({mode})", model_rnd, assets_rnd))

    plot_pred_vs_truth_grid(
        plot_configs=ranking_plot_configs,
        ds_test=ds_sjm_test,
        device=device,
        cfg=cfg,
        ncols=4,
        ax_xlim=(-8, 6),
        ax_ylim=(-8, 6),
        title=f"Closest vs random ({mode}) → SJM",
        save_path=f"figures/ranking_sjm_{mode}_pred_vs_truth",
        figsize=(25, 12),
    )

## Fine tuning:

In [ ]:
from torch.utils.data import Subset


def make_winter_only_loaders(ds_ft,
                             ds_pooled_scaler,
                             train_idx,
                             val_idx,
                             cfg,
                             batch_size=64):
    """
    Apply pooled scalers to ds_ft, then return loaders containing
    only winter sequences.
    """
    ds_ft_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_ft)
    ds_ft_copy.set_scalers_from(ds_pooled_scaler)
    ds_ft_copy.transform_inplace()

    # filter train_idx to winter only
    iw = ds_ft_copy.iw.numpy()
    train_idx_w = train_idx[iw[train_idx]]
    val_idx_w = val_idx[iw[val_idx]]

    print(
        f"  Winter-only sequences: {len(train_idx_w)} train | {len(val_idx_w)} val"
    )

    if len(train_idx_w) == 0 or len(val_idx_w) == 0:
        raise ValueError("No winter sequences in train or val split — "
                         "try a larger FT fraction.")

    train_dl = DataLoader(
        Subset(ds_ft_copy, train_idx_w),
        batch_size=batch_size,
        shuffle=True,
    )
    val_dl = DataLoader(
        Subset(ds_ft_copy, val_idx_w),
        batch_size=batch_size,
        shuffle=False,
    )
    return train_dl, val_dl

In [ ]:
def finetune_transformer_on_target(
    cfg,
    model_pooled,
    ds_ft,
    ds_pooled_scaler,
    device,
    best_params,
    model_filename,
    strategy="adapter",
    epochs=50,
    lr_factor=0.1,
    force_retrain=False,
):
    import copy
    assert strategy in (
        "adapter",
        "head_only",
        "full",
        "adapter_winter",
        "head_only_winter",  # <-- new
    )

    model_ft = copy.deepcopy(model_pooled)

    if not force_retrain and os.path.exists(model_filename):
        print(f"Loading [{strategy}] from {model_filename}")
        model_ft.load_state_dict(
            torch.load(model_filename, map_location=device))
        return model_ft

    # --- split indices ---
    train_idx_ft, val_idx_ft = mbm.data_processing.MBSequenceDataset.split_indices(
        len(ds_ft), val_ratio=0.2, seed=cfg.seed)

    # --- build loaders: winter-only or all ---
    winter_only = strategy.endswith("_winter")

    if winter_only:
        train_dl, val_dl = make_winter_only_loaders(
            ds_ft,
            ds_pooled_scaler,
            train_idx_ft,
            val_idx_ft,
            cfg,
            batch_size=64,
        )
    else:
        ds_ft_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ds_ft)
        ds_ft_copy.set_scalers_from(ds_pooled_scaler)
        ds_ft_copy.transform_inplace()
        train_dl, val_dl = ds_ft_copy.make_loaders(
            train_idx=train_idx_ft,
            val_idx=val_idx_ft,
            fit_and_transform=False,
            seed=cfg.seed,
            use_weighted_sampler=True,
            verbose=True,
        )

    # --- freeze/unfreeze ---
    base_strategy = strategy.replace("_winter", "")

    if base_strategy == "head_only":
        for param in model_ft.parameters():
            param.requires_grad = False
        head_modules = ([model_ft.head_w, model_ft.head_a]
                        if model_ft.two_heads else [model_ft.head])
        for m in head_modules:
            for param in m.parameters():
                param.requires_grad = True

    elif base_strategy == "adapter":
        for param in model_ft.parameters():
            param.requires_grad = False
        if not model_ft.use_adapter:
            fused_dim = (model_ft.head_w.in_features
                         if model_ft.two_heads else model_ft.head.in_features)
            model_ft.adapter = mbm.models.BottleneckAdapter(
                dim=fused_dim,
                bottleneck=int(best_params.get("adapter_bottleneck", 32)),
                dropout=float(best_params.get("adapter_dropout", 0.0)),
                use_ln=bool(best_params.get("adapter_use_ln", True)),
            ).to(device)
            model_ft.use_adapter = True
        for param in model_ft.adapter.parameters():
            param.requires_grad = True

    # "full" -> all params trainable

    n_trainable = sum(p.numel() for p in model_ft.parameters()
                      if p.requires_grad)
    n_total = sum(p.numel() for p in model_ft.parameters())
    print(f"  [{strategy}] {n_trainable} / {n_total} params trainable  "
          f"({'winter only' if winter_only else 'all seasons'})")

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model_ft.parameters()),
        lr=best_params['lr'] * lr_factor,
        weight_decay=best_params['weight_decay'],
    )
    loss_fn = mbm.models.Transformer_MB.resolve_loss_fn(best_params)

    history, best_val, _ = model_ft.train_loop(
        device=device,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=epochs,
        optimizer=optimizer,
        loss_fn=loss_fn,
        es_patience=10,
        es_min_delta=1e-4,
        sched_factor=0.5,
        sched_patience=4,
        sched_min_lr=1e-7,
        log_every=5,
        verbose=True,
        save_best_path=model_filename,
    )

    plot_history_lstm(history)
    model_ft.load_state_dict(torch.load(model_filename, map_location=device))
    return model_ft

### FT strategies:

In [ ]:
PCT = 20

# ================================================================
# 0. SJM ID split (stratified by period)
# ================================================================
SJM_FT_FRAC = 0.10

df_sjm = res_xreg_by_source['SJM']['data_monthly']
df_sjm_aug = res_xreg_by_source['SJM']['data_monthly_aug']

winter_ids = df_sjm[df_sjm['PERIOD'] == 'winter']['ID'].unique().tolist()
annual_ids = df_sjm[df_sjm['PERIOD'] == 'annual']['ID'].unique().tolist()

rng = np.random.default_rng(cfg.seed)
rng.shuffle(winter_ids)
rng.shuffle(annual_ids)

n_ft_w = max(1, int(len(winter_ids) * SJM_FT_FRAC))
n_ft_a = max(1, int(len(annual_ids) * SJM_FT_FRAC))

sjm_ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
sjm_test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

print(f"SJM FT  : {n_ft_w} winter + {n_ft_a} annual = {len(sjm_ft_ids)} IDs")
print(f"SJM test: {len(winter_ids)-n_ft_w} winter + "
      f"{len(annual_ids)-n_ft_a} annual = {len(sjm_test_ids)} IDs")

ds_sjm_ft = build_combined_LSTM_dataset(
    df_loss=df_sjm[df_sjm['ID'].isin(sjm_ft_ids)],
    df_full=df_sjm_aug[df_sjm_aug['ID'].isin(sjm_ft_ids)],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['SJM']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['SJM']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)

ds_sjm_test_holdout = build_combined_LSTM_dataset(
    df_loss=df_sjm[df_sjm['ID'].isin(sjm_test_ids)],
    df_full=df_sjm_aug[df_sjm_aug['ID'].isin(sjm_test_ids)],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['SJM']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['SJM']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)

print(
    f"\nSJM FT dataset   : {len(ds_sjm_ft)} sequences "
    f"({ds_sjm_ft.iw.sum().item()} winter | {ds_sjm_ft.ia.sum().item()} annual)"
)
print(f"SJM test holdout : {len(ds_sjm_test_holdout)} sequences "
      f"({ds_sjm_test_holdout.iw.sum().item()} winter | "
      f"{ds_sjm_test_holdout.ia.sum().item()} annual)")
print(f"Winter sequences in FT : {ds_sjm_ft.iw.sum().item()}")

# ================================================================
# 1. Load cached assets for 20% closest (joint ranking)
# ================================================================
assets_20pct = build_assets_from_glacier_list(
    glaciers=gl_subsets_sjm_joint[PCT]['closest'],  # <-- use joint alias
    df_ranked=df_ranked_sjm_joint,
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR_RANK /
                   f"assets_joint_{PCT}pct_closest.joblib"  # <-- mode prefix
                   ),
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    force_recompute=False,
    src_regions=SOURCE_REGIONS,
)

# ================================================================
# 2. Load pretrained transformer from joint subfolder
# ================================================================
model_pretrained, _, _ = train_or_load_one_source_model(
    cfg=cfg,
    key=f"{PCT}pct_closest",
    lstm_assets=assets_20pct,
    best_params=best_params_gs,
    device=device,
    models_dir=BASE_DIR / "models/ranking_sjm" / "joint",  # <-- subfolder
    prefix="transformer_rank",
    train_flag=False,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=False,
)

# ================================================================
# 3. Scaler donor from European training data
# ================================================================
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_20pct["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_20pct["train_idx"],
    val_idx=assets_20pct["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

# ================================================================
# 4. Fine-tune on SJM FT IDs
# ================================================================
ft_models = {}
for strategy in [
        "head_only", "adapter", "full", "head_only_winter", "adapter_winter"
]:
    ft_models[strategy] = finetune_transformer_on_target(
        cfg=cfg,
        model_pooled=model_pretrained,
        ds_ft=ds_sjm_ft,
        ds_pooled_scaler=ds_pooled_scaler,
        device=device,
        best_params=best_params_gs,
        model_filename=str(
            BASE_DIR / "models/ranking_sjm" / "joint" /  # <-- subfolder
            f"transformer_ft_{strategy}_sjm_{PCT}pct_{MODEL_DATE}.pt"),
        strategy=strategy,
        epochs=50,
        lr_factor=0.1,
        force_retrain=TRAIN_FLAG,
    )
    print(f"\n{strategy} fine-tuning done")

# ================================================================
# 5. Evaluate on SJM holdout
# ================================================================
print(f"\n{'='*60}")
print("Fine-tuning results on SJM test holdout (90% of IDs)")
print(f"{'='*60}")


def _eval_on_sjm_holdout(model, label):
    ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_20pct["ds_train"])
    ds_scaler.make_loaders(
        train_idx=assets_20pct["train_idx"],
        val_idx=assets_20pct["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_sjm_test_holdout)
    sjm_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy,
        ds_scaler,
        batch_size=128,
        seed=cfg.seed,
    )
    m, _ = model.evaluate_with_preds(device, sjm_dl, ds_copy)
    print(f"  {label:25s}  "
          f"RMSE_a={m['RMSE_annual']:.3f}  "
          f"RMSE_w={m['RMSE_winter']:.3f}  "
          f"R2_a={m['R2_annual']:.3f}  "
          f"R2_w={m['R2_winter']:.3f}")
    return m


metrics_base = _eval_on_sjm_holdout(model_pretrained, "no_ft")

ft_metrics = {}
for strategy, model_ft in ft_models.items():
    ft_metrics[strategy] = _eval_on_sjm_holdout(model_ft, strategy)

In [ ]:
FRACS_TO_PLOT = [10, 20]
FT_STRATEGIES = ["head_only", "full", "adapter"]

ranking_plot_configs = []

for pct in FRACS_TO_PLOT:
    # --- closest (no FT) ---
    assets_close = build_assets_from_glacier_list(
        glaciers=gl_subsets_sjm_joint[pct]['closest'],
        df_ranked=df_ranked_sjm_joint,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(CACHE_DIR_RANK /
                       f"assets_joint_{pct}pct_closest.joblib"),
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        force_recompute=False,
        src_regions=SOURCE_REGIONS,
    )
    model_close, _, _ = train_or_load_one_source_model(
        cfg=cfg,
        key=f"{pct}pct_closest",
        lstm_assets=assets_close,
        best_params=best_params_gs,
        device=device,
        models_dir=BASE_DIR / "models/ranking_sjm" / "joint",
        prefix="transformer_rank",
        train_flag=False,
        date=MODEL_DATE,
        model_type="transformer",
        verbose=False,
    )
    ranking_plot_configs.append(
        (f"Closest {pct}% (no FT)", model_close, assets_close))

    # --- random seed 0 (no FT) ---
    assets_rnd = build_assets_from_glacier_list(
        glaciers=gl_subsets_sjm_joint[pct]['random'][0]['glaciers'],
        df_ranked=df_ranked_sjm_joint,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(CACHE_DIR_RANK /
                       f"assets_joint_{pct}pct_random_0.joblib"),
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        force_recompute=False,
        src_regions=SOURCE_REGIONS,
    )
    model_rnd, _, _ = train_or_load_one_source_model(
        cfg=cfg,
        key=f"{pct}pct_random_0",
        lstm_assets=assets_rnd,
        best_params=best_params_gs,
        device=device,
        models_dir=BASE_DIR / "models/ranking_sjm" / "joint",
        prefix="transformer_rank",
        train_flag=False,
        date=MODEL_DATE,
        model_type="transformer",
        verbose=False,
    )
    ranking_plot_configs.append(
        (f"Random {pct}% (no FT, seed 0)", model_rnd, assets_rnd))

    # --- fine-tuned strategies (only for PCT=20) ---
    if pct == PCT:
        for strategy in FT_STRATEGIES:
            model_ft = ft_models.get(strategy)
            if model_ft is not None:
                ranking_plot_configs.append((f"FT {strategy} ({pct}% closest)",
                                             model_ft, assets_close))

fig = plot_pred_vs_truth_grid(
    plot_configs=ranking_plot_configs,
    ds_test=ds_sjm_test_holdout,  # holdout, not full SJM
    device=device,
    cfg=cfg,
    ncols=4,
    ax_xlim=(-8, 6),
    ax_ylim=(-8, 6),
    figsize=(25, 12),
    title="Closest vs random vs FT strategies → SJM holdout",
    save_path="figures/ranking_sjm_ft_pred_vs_truth",
)

### FT fractions:

In [ ]:
SJM_FT_FRACS = [0.05, 0.10, 0.15, 0.20]

ft_models_by_frac = {}
ds_sjm_ft_by_frac = {}
ds_sjm_holdout_by_frac = {}

for SJM_FT_FRAC in SJM_FT_FRACS:
    pct_label = int(SJM_FT_FRAC * 100)
    print(f"\n{'='*60}")
    print(f"  SJM FT fraction: {pct_label}%")

    winter_ids = df_sjm[df_sjm['PERIOD'] == 'winter']['ID'].unique().tolist()
    annual_ids = df_sjm[df_sjm['PERIOD'] == 'annual']['ID'].unique().tolist()

    rng = np.random.default_rng(cfg.seed)
    rng.shuffle(winter_ids)
    rng.shuffle(annual_ids)

    n_ft_w = max(1, int(len(winter_ids) * SJM_FT_FRAC))
    n_ft_a = max(1, int(len(annual_ids) * SJM_FT_FRAC))

    sjm_ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
    sjm_test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

    print(f"  FT  : {n_ft_w} winter + {n_ft_a} annual = {len(sjm_ft_ids)} IDs")
    print(f"  Test: {len(winter_ids)-n_ft_w} winter + "
          f"{len(annual_ids)-n_ft_a} annual = {len(sjm_test_ids)} IDs")

    ds_ft = build_combined_LSTM_dataset(
        df_loss=df_sjm[df_sjm['ID'].isin(sjm_ft_ids)],
        df_full=df_sjm_aug[df_sjm_aug['ID'].isin(sjm_ft_ids)],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        months_head_pad=res_xreg_by_source['SJM']['months_head_pad'],
        months_tail_pad=res_xreg_by_source['SJM']['months_tail_pad'],
        normalize_target=True,
        expect_target=True,
        show_progress=False,
    )
    ds_holdout = build_combined_LSTM_dataset(
        df_loss=df_sjm[df_sjm['ID'].isin(sjm_test_ids)],
        df_full=df_sjm_aug[df_sjm_aug['ID'].isin(sjm_test_ids)],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        months_head_pad=res_xreg_by_source['SJM']['months_head_pad'],
        months_tail_pad=res_xreg_by_source['SJM']['months_tail_pad'],
        normalize_target=True,
        expect_target=True,
        show_progress=False,
    )

    print(f"  FT dataset : {len(ds_ft)} sequences "
          f"({ds_ft.iw.sum().item()} winter | {ds_ft.ia.sum().item()} annual)")
    print(
        f"  Holdout    : {len(ds_holdout)} sequences "
        f"({ds_holdout.iw.sum().item()} winter | {ds_holdout.ia.sum().item()} annual)"
    )

    ds_sjm_ft_by_frac[pct_label] = ds_ft
    ds_sjm_holdout_by_frac[pct_label] = ds_holdout

    model_ft = finetune_transformer_on_target(
        cfg=cfg,
        model_pooled=model_pretrained,
        ds_ft=ds_ft,
        ds_pooled_scaler=ds_pooled_scaler,
        device=device,
        best_params=best_params_gs,
        model_filename=str(
            BASE_DIR / "models/ranking_sjm" / "joint" /  # <-- subfolder
            f"transformer_ft_full_sjm_{pct_label}pct_sjmids_{MODEL_DATE}.pt"),
        strategy="full",
        epochs=50,
        lr_factor=0.1,
        force_retrain=TRAIN_FLAG,
    )
    ft_models_by_frac[pct_label] = model_ft

# --- evaluate all fractions on fixed holdout ---
print(f"\n{'='*60}")
print("Full FT results by SJM fraction → SJM holdout")
print(f"{'='*60}")

_eval_on_sjm_holdout(model_pretrained, "no_ft (0%)")

for pct_label, model_ft in ft_models_by_frac.items():
    ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_20pct["ds_train"])
    ds_scaler.make_loaders(
        train_idx=assets_20pct["train_idx"],
        val_idx=assets_20pct["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_sjm_test_holdout)  # <-- fixed holdout, not per-fraction
    sjm_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy,
        ds_scaler,
        batch_size=128,
        seed=cfg.seed,
    )
    m, _ = model_ft.evaluate_with_preds(device, sjm_dl, ds_copy)
    print(f"  full FT {pct_label:3d}% SJM IDs  "
          f"RMSE_a={m['RMSE_annual']:.3f}  "
          f"RMSE_w={m['RMSE_winter']:.3f}  "
          f"R2_a={m['R2_annual']:.3f}  "
          f"R2_w={m['R2_winter']:.3f}")

# --- plot ---
FRACS_TO_PLOT = [10, 20]
FT_STRATEGIES = ["head_only", "full", "adapter"]
SJM_FT_FRACS_PLOT = [5, 10, 15, 20]

ranking_plot_configs = []

for pct in FRACS_TO_PLOT:
    # --- closest (no FT) ---
    assets_close = build_assets_from_glacier_list(
        glaciers=gl_subsets_sjm_joint[pct]['closest'],
        df_ranked=df_ranked_sjm_joint,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(CACHE_DIR_RANK /
                       f"assets_joint_{pct}pct_closest.joblib"),
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        force_recompute=False,
        src_regions=SOURCE_REGIONS,
    )
    model_close, _, _ = train_or_load_one_source_model(
        cfg=cfg,
        key=f"{pct}pct_closest",
        lstm_assets=assets_close,
        best_params=best_params_gs,
        device=device,
        models_dir=BASE_DIR / "models/ranking_sjm" / "joint",
        prefix="transformer_rank",
        train_flag=False,
        date=MODEL_DATE,
        model_type="transformer",
        verbose=False,
    )
    ranking_plot_configs.append(
        (f"Closest {pct}% (no FT)", model_close, assets_close))

    # --- random seed 0 (no FT) ---
    assets_rnd = build_assets_from_glacier_list(
        glaciers=gl_subsets_sjm_joint[pct]['random'][0]['glaciers'],
        df_ranked=df_ranked_sjm_joint,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(CACHE_DIR_RANK /
                       f"assets_joint_{pct}pct_random_0.joblib"),
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        force_recompute=False,
        src_regions=SOURCE_REGIONS,
    )
    model_rnd, _, _ = train_or_load_one_source_model(
        cfg=cfg,
        key=f"{pct}pct_random_0",
        lstm_assets=assets_rnd,
        best_params=best_params_gs,
        device=device,
        models_dir=BASE_DIR / "models/ranking_sjm" / "joint",
        prefix="transformer_rank",
        train_flag=False,
        date=MODEL_DATE,
        model_type="transformer",
        verbose=False,
    )
    ranking_plot_configs.append(
        (f"Random {pct}% (no FT, seed 0)", model_rnd, assets_rnd))

# --- no-FT baseline ---
ranking_plot_configs.append(("no FT (0% SJM)", model_pretrained, assets_20pct))

# --- FT strategies at 10% SJM IDs ---
for strategy in FT_STRATEGIES:
    model_ft = ft_models.get(strategy)
    if model_ft is not None:
        ranking_plot_configs.append(
            (f"FT {strategy} (10% SJM IDs)", model_ft, assets_20pct))

# --- full FT by SJM fraction ---
for sjm_pct in SJM_FT_FRACS_PLOT:
    model_ft = ft_models_by_frac.get(sjm_pct)
    if model_ft is not None:
        ranking_plot_configs.append(
            (f"Transformer Full FT ({sjm_pct}% SJM IDs)", model_ft, assets_20pct))

fig = plot_pred_vs_truth_grid(
    plot_configs=ranking_plot_configs,
    ds_test=ds_sjm_test_holdout,
    device=device,
    cfg=cfg,
    ncols=4,
    ax_xlim=(-8, 6),
    ax_ylim=(-8, 6),
    figsize=(25, 12),
    title="Closest vs random vs FT strategies → SJM holdout",
    save_path="figures/ranking_sjm_ft_pred_vs_truth",
)

In [ ]:
# ============================================================
# Regional LSTM baselines + LSTM fine-tuning for comparison
# ============================================================

LSTM_PRETRAINED_CKPT = str(
    BASE_DIR / "models" /
    "ranking_sjm/joint/lstm_rank_100pct_full_2026-05-21.pt")

lstm_ft_results = {}
lstm_ft_models_by_frac = {}  # store models for plotting

for SJM_FT_FRAC in SJM_FT_FRACS:
    pct_label = int(SJM_FT_FRAC * 100)
    ds_ft = ds_sjm_ft_by_frac[pct_label]

    n = len(ds_ft)
    idx = list(range(n))
    split = int(0.8 * n)
    print(f"\n  LSTM full FT {pct_label}%: {split} train / {n-split} val sequences")

    # --- build loaders from SJM FT data, fitting scalers on it ---
    ds_ft_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(ds_ft)
    train_dl, val_dl = ds_ft_copy.make_loaders(
        train_idx=idx[:split],
        val_idx=idx[split:],
        batch_size_train=64,
        batch_size_val=128,
        seed=cfg.seed,
        fit_and_transform=True,
        shuffle_train=True,
        use_weighted_sampler=True,
        verbose=False,
    )

    # --- load pretrained LSTM weights ---
    model_lstm_ft = mbm.models.LSTM_MB.build_model_from_params(
        cfg, lstm_params, device, verbose=False)
    state = torch.load(LSTM_PRETRAINED_CKPT, map_location=device)
    model_lstm_ft.load_state_dict(state, strict=True)
    loss_fn = mbm.models.LSTM_MB.resolve_loss_fn(lstm_params)

    # --- full fine-tune (all layers, low lr) ---
    unfreeze_all(model_lstm_ft)
    out_path = str(CACHE_DIR_RANK / f"lstm_ft_full_sjm_{pct_label}pct_{MODEL_DATE}.pt")

    if TRAIN_FLAG or not os.path.exists(out_path):
        opt = torch.optim.AdamW(
            model_lstm_ft.parameters(),
            lr=1e-5,
            weight_decay=lstm_params["weight_decay"],
        )
        history, best_val, _ = model_lstm_ft.train_loop(
            device=device,
            train_dl=train_dl,
            val_dl=val_dl,
            epochs=80,
            optimizer=opt,
            clip_val=1.0,
            loss_fn=loss_fn,
            es_patience=10,
            save_best_path=out_path,
            verbose=False,
        )
    state = torch.load(out_path, map_location=device)
    model_lstm_ft.load_state_dict(state)
    lstm_ft_models_by_frac[pct_label] = model_lstm_ft  # store for plotting

    # --- evaluate on FIXED holdout (same as Transformer FT) ---
    ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_20pct["ds_train"])
    ds_scaler.make_loaders(
        train_idx=assets_20pct["train_idx"],
        val_idx=assets_20pct["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_sjm_test_holdout)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, ds_scaler, batch_size=128, seed=cfg.seed,
    )
    m_ft, _ = model_lstm_ft.evaluate_with_preds(device, test_dl, ds_copy)
    lstm_ft_results[pct_label] = m_ft
    print(f"  LSTM full FT {pct_label}%: "
          f"RMSE_a={m_ft['RMSE_annual']:.3f}  R²_a={m_ft['R2_annual']:.3f}")


# --- plot: append LSTM FT to existing ranking_plot_configs ---
for sjm_pct in SJM_FT_FRACS_PLOT:
    model_lstm_ft = lstm_ft_models_by_frac.get(sjm_pct)
    if model_lstm_ft is not None:
        ranking_plot_configs.append(
            (f"LSTM full FT ({sjm_pct}% SJM IDs)", model_lstm_ft, assets_20pct))

fig = plot_pred_vs_truth_grid(
    plot_configs=ranking_plot_configs,
    ds_test=ds_sjm_test_holdout,
    device=device,
    cfg=cfg,
    ncols=4,
    ax_xlim=(-8, 6),
    ax_ylim=(-8, 6),
    figsize=(25, 16),
    title="Closest vs random vs FT strategies → SJM holdout",
    save_path="figures/ranking_sjm_ft_pred_vs_truth_with_lstm",
)

In [ ]:
# ============================================================
# Regional LSTM trained from scratch on X% SJM data
# ============================================================

lstm_scratch_results = {}
lstm_scratch_models_by_frac = {}  # store for plotting

for SJM_FT_FRAC in SJM_FT_FRACS:
    pct_label = int(SJM_FT_FRAC * 100)
    ds_ft = ds_sjm_ft_by_frac[pct_label]

    n = len(ds_ft)
    idx = list(range(n))
    split = int(0.8 * n)
    print(f"\n  LSTM scratch {pct_label}%: {split} train / {n-split} val sequences")

    # --- build loaders ---
    ds_ft_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(ds_ft)
    train_dl, val_dl = ds_ft_copy.make_loaders(
        train_idx=idx[:split],
        val_idx=idx[split:],
        batch_size_train=64,
        batch_size_val=128,
        seed=cfg.seed,
        fit_and_transform=True,
        shuffle_train=True,
        use_weighted_sampler=True,
        verbose=False,
    )

    # --- build model from scratch (no pretrained weights) ---
    model_lstm_scratch = mbm.models.LSTM_MB.build_model_from_params(
        cfg, lstm_params, device, verbose=False)
    loss_fn = mbm.models.LSTM_MB.resolve_loss_fn(lstm_params)

    out_path = str(CACHE_DIR_RANK / f"lstm_scratch_sjm_{pct_label}pct_{MODEL_DATE}.pt")

    if TRAIN_FLAG or not os.path.exists(out_path):
        history, best_val, _ = model_lstm_scratch.train_loop(
            device=device,
            train_dl=train_dl,
            val_dl=val_dl,
            epochs=150,              # more epochs than FT since training from scratch
            lr=lstm_params["lr"],    # full lr, not reduced like FT
            weight_decay=lstm_params["weight_decay"],
            clip_val=1.0,
            sched_factor=0.5,
            sched_patience=6,
            sched_threshold=0.01,
            sched_threshold_mode="rel",
            sched_cooldown=1,
            sched_min_lr=1e-6,
            loss_fn=loss_fn,
            es_patience=15,
            es_min_delta=1e-4,
            log_every=5,
            verbose=False,
            save_best_path=out_path,
        )
    state = torch.load(out_path, map_location=device)
    model_lstm_scratch.load_state_dict(state)
    lstm_scratch_models_by_frac[pct_label] = model_lstm_scratch  # store for plotting

    # --- evaluate on FIXED holdout (same as everything else) ---
    ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_20pct["ds_train"])
    ds_scaler.make_loaders(
        train_idx=assets_20pct["train_idx"],
        val_idx=assets_20pct["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_sjm_test_holdout)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, ds_scaler, batch_size=128, seed=cfg.seed,
    )
    m_scratch, _ = model_lstm_scratch.evaluate_with_preds(device, test_dl, ds_copy)
    lstm_scratch_results[pct_label] = m_scratch
    print(f"  LSTM scratch {pct_label}%: "
          f"RMSE_a={m_scratch['RMSE_annual']:.3f}  R²_a={m_scratch['R2_annual']:.3f}")


# --- append to ranking_plot_configs and replot ---
for sjm_pct in SJM_FT_FRACS_PLOT:
    model_scratch = lstm_scratch_models_by_frac.get(sjm_pct)
    if model_scratch is not None:
        ranking_plot_configs.append(
            (f"LSTM scratch ({sjm_pct}% SJM IDs)", model_scratch, assets_20pct))

fig = plot_pred_vs_truth_grid(
    plot_configs=ranking_plot_configs,
    ds_test=ds_sjm_test_holdout,
    device=device,
    cfg=cfg,
    ncols=4,
    ax_xlim=(-8, 6),
    ax_ylim=(-8, 6),
    figsize=(25, 20),              # taller to fit 5th row
    title="Closest vs random vs FT strategies → SJM holdout",
    save_path="figures/ranking_sjm_ft_pred_vs_truth_with_lstm_scratch",
)

In [ ]:
# ============================================================
# Transformer trained from scratch on X% SJM data
# ============================================================

transformer_scratch_results = {}
transformer_scratch_models_by_frac = {}

for SJM_FT_FRAC in SJM_FT_FRACS:
    pct_label = int(SJM_FT_FRAC * 100)
    ds_ft = ds_sjm_ft_by_frac[pct_label]

    n = len(ds_ft)
    idx = list(range(n))
    split = int(0.8 * n)
    print(f"\n  Transformer scratch {pct_label}%: {split} train / {n-split} val sequences")

    lstm_assets_scratch = {
        "ds_train":  ds_ft,
        "train_idx": idx[:split],
        "val_idx":   idx[split:],
    }

    model_tf_scratch, _, _ = train_or_load_one_source_model(
        cfg=cfg,
        key=f"sjm_tf_scratch_{pct_label}pct",
        lstm_assets=lstm_assets_scratch,
        best_params=best_params_gs,
        device=device,
        models_dir=str(CACHE_DIR_RANK),
        prefix="transformer_scratch_sjm",
        train_flag=TRAIN_FLAG,
        model_type="transformer",
        date=MODEL_DATE,
        verbose=False,
    )
    transformer_scratch_models_by_frac[pct_label] = model_tf_scratch

    # --- evaluate on FIXED holdout ---
    ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_20pct["ds_train"])
    ds_scaler.make_loaders(
        train_idx=assets_20pct["train_idx"],
        val_idx=assets_20pct["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_sjm_test_holdout)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, ds_scaler, batch_size=128, seed=cfg.seed,
    )
    m_tf_scratch, _ = model_tf_scratch.evaluate_with_preds(device, test_dl, ds_copy)
    transformer_scratch_results[pct_label] = m_tf_scratch
    print(f"  Transformer scratch {pct_label}%: "
          f"RMSE_a={m_tf_scratch['RMSE_annual']:.3f}  R²_a={m_tf_scratch['R2_annual']:.3f}")


# ============================================================
# Figure 1: Fine-tuning comparison (Transformer FT vs LSTM FT)
# ============================================================
ft_plot_configs = []

# --- baselines ---
ft_plot_configs.append(("no FT (0% SJM)", model_pretrained, assets_20pct))

for sjm_pct in SJM_FT_FRACS_PLOT:
    model_tf_ft = ft_models_by_frac.get(sjm_pct)
    if model_tf_ft is not None:
        ft_plot_configs.append(
            (f"Transformer FT ({sjm_pct}% SJM IDs)", model_tf_ft, assets_20pct))

for sjm_pct in SJM_FT_FRACS_PLOT:
    model_lstm_ft = lstm_ft_models_by_frac.get(sjm_pct)
    if model_lstm_ft is not None:
        ft_plot_configs.append(
            (f"LSTM FT ({sjm_pct}% SJM IDs)", model_lstm_ft, assets_20pct))

fig1 = plot_pred_vs_truth_grid(
    plot_configs=ft_plot_configs,
    ds_test=ds_sjm_test_holdout,
    device=device,
    cfg=cfg,
    ncols=4,
    ax_xlim=(-8, 6),
    ax_ylim=(-8, 6),
    figsize=(25, 12),
    title="Fine-tuning comparison: Transformer FT vs LSTM FT → SJM holdout",
    save_path="figures/ranking_sjm_finetuning_comparison",
)


# ============================================================
# Figure 2: Scratch comparison (Transformer scratch vs LSTM scratch)
# ============================================================
scratch_plot_configs = []

for sjm_pct in SJM_FT_FRACS_PLOT:
    model_tf_scratch = transformer_scratch_models_by_frac.get(sjm_pct)
    if model_tf_scratch is not None:
        scratch_plot_configs.append(
            (f"Transformer scratch ({sjm_pct}% SJM IDs)", model_tf_scratch, assets_20pct))

for sjm_pct in SJM_FT_FRACS_PLOT:
    model_lstm_scratch = lstm_scratch_models_by_frac.get(sjm_pct)
    if model_lstm_scratch is not None:
        scratch_plot_configs.append(
            (f"LSTM scratch ({sjm_pct}% SJM IDs)", model_lstm_scratch, assets_20pct))

fig2 = plot_pred_vs_truth_grid(
    plot_configs=scratch_plot_configs,
    ds_test=ds_sjm_test_holdout,
    device=device,
    cfg=cfg,
    ncols=4,
    ax_xlim=(-8, 6),
    ax_ylim=(-8, 6),
    figsize=(25, 8),
    title="Scratch comparison: Transformer scratch vs LSTM scratch → SJM holdout",
    save_path="figures/ranking_sjm_scratch_comparison",
)